# Частина 1
## Завдання №1
Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних;

In [2]:
import pandas as pd
import urllib.request
import os
from datetime import datetime
os.makedirs("VHI_data",exist_ok=True)
def download_all_files():
    for provinceID in range(1,28):
        url=f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={provinceID}&year1=1981&year2=2024&type=Mean"
        time=datetime.now().strftime("%Y%m%d")
        file_name=f"vhi_province_{provinceID}_{time}.csv"
        file_path=os.path.join("VHI_data",file_name)
        if not os.path.exists(file_path):
            urllib.request.urlretrieve(url, file_path)
            print(f"{provinceID}.Завантажено область{provinceID}")
        else:
            print(f"{provinceID}.Область{provinceID} вже є")
download_all_files()

1.Завантажено область1
2.Завантажено область2
3.Завантажено область3
4.Завантажено область4
5.Завантажено область5
6.Завантажено область6
7.Завантажено область7
8.Завантажено область8
9.Завантажено область9
10.Завантажено область10
11.Завантажено область11
12.Завантажено область12
13.Завантажено область13
14.Завантажено область14
15.Завантажено область15
16.Завантажено область16
17.Завантажено область17
18.Завантажено область18
19.Завантажено область19
20.Завантажено область20
21.Завантажено область21
22.Завантажено область22
23.Завантажено область23
24.Завантажено область24
25.Завантажено область25
26.Завантажено область26
27.Завантажено область27


## Завдання №2
Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області

In [3]:
from io import StringIO
province_names={
    1: "Черкаська", 2: "Чернігівська", 3: "Чернівецька", 4: "Республіка Крим",
    5: "Дніпропетровська", 6: "Донецька", 7: "Івано-Франківська", 8: "Харківська",
    9: "Херсонська", 10: "Хмельницька", 11: "Київська", 12: "Київ",
    13: "Кіровоградська", 14: "Луганська", 15: "Львівська", 16: "Миколаївська",
    17: "Одеська", 18: "Полтавська", 19: "Рівненська", 20: "Севастополь",
    21: "Сумська", 22: "Тернопільська", 23: "Закарпатська", 24: "Вінницька",
    25: "Волинська", 26: "Запорізька", 27: "Житомирська"
}
def read_all_files_and_cleaning_data(file_path="VHI_data"):
    all_data=[]
    for filename in os.listdir(file_path):
        if filename.endswith(".csv"):
            filepath = os.path.join(file_path,filename)
            with open(filepath,'r') as f:
                lines=f.readlines()
            clean_data=[]
            for line in lines:
                if line.strip():
                    clean_data.append(line.rstrip(',\n'))
            data_str='\n'.join(clean_data)
            df=pd.read_csv(StringIO(data_str),skiprows=1)
            df.columns=['Year','Week','SMN','SMT','VCI','TCI','VHI']
            df['Year']=df['Year'].astype(str).str.replace('<tt><pre>','',regex=False)
            province_id = int(filename.split('_')[2])
            df['province_id'] = province_id
            df['Province_Name']=province_names.get(province_id)
            all_data.append(df)
    final_df=pd.concat(all_data, ignore_index=True)
    final_df['Year']=pd.to_numeric(final_df['Year'],errors='coerce')
    final_df = final_df.dropna(subset=['Year'])
    final_df['Year'] = final_df['Year'].astype(int)
    return final_df
df=read_all_files_and_cleaning_data()
print(df.head())

   Year  Week    SMN     SMT    VCI    TCI    VHI  province_id Province_Name
0  1982   1.0  0.059  258.24  51.11  48.78  49.95           10   Хмельницька
1  1982   2.0  0.063  261.53  55.89  38.20  47.04           10   Хмельницька
2  1982   3.0  0.063  263.45  57.30  32.69  44.99           10   Хмельницька
3  1982   4.0  0.061  265.10  53.96  28.62  41.29           10   Хмельницька
4  1982   5.0  0.058  266.42  46.87  28.57  37.72           10   Хмельницька


## Завдання №3
Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька).

In [4]:
def reindex(dataframe):
    old_to_new_id = {1: 22, 2: 24, 3: 23, 4: 27, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
        11: 9, 12: 25, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16,
        20: 26, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5}
    dataframe['new_province_id'] = dataframe['province_id'].map(old_to_new_id)
    new_province_names={
        1: "Вінницька", 2: "Волинська", 3: "Дніпропетровська", 4: "Донецька", 5: "Житомирська",
        6: "Закарпатська", 7: "Запорізька", 8: "Івано-Франківська", 9: "Київська", 10: "Кіровоградська",
        11: "Луганська", 12: "Львівська", 13: "Миколаївська", 14: "Одеська", 15: "Полтавська",
        16: "Рівненська", 17: "Сумська", 18: "Тернопільська", 19: "Харківська", 20: "Херсонська",
        21: "Хмельницька", 22: "Черкаська", 23: "Чернівецька", 24: "Чернігівська", 25: "м. Київ",
        26: "Севастополь", 27: "Республіка Крим"}
    dataframe['new_province_name']=dataframe['new_province_id'].map(new_province_names)
    return dataframe
df = reindex(df)
print(df[['Year', 'Week', 'VHI', 'province_id', 'Province_Name', 'new_province_id']].head())

   Year  Week    VHI  province_id Province_Name  new_province_id
0  1982   1.0  49.95           10   Хмельницька               21
1  1982   2.0  47.04           10   Хмельницька               21
2  1982   3.0  44.99           10   Хмельницька               21
3  1982   4.0  41.29           10   Хмельницька               21
4  1982   5.0  37.72           10   Хмельницька               21


## Завдання №4.1
Реалізувати процедури для формування вибірок наступного виду: Ряд VHI для області за вказаний рік

In [5]:
def get_vhi_year(dataframe,province_id,year):
    condition1=dataframe['new_province_id']==province_id
    condition2=dataframe['Year']==year
    filtered_data=dataframe[condition1 & condition2]
    return filtered_data[['Week', 'VHI']]
print("Хмельницька область за 1982 рік:")
result=get_vhi_year(df,21,1982)
print(result.head(10))

Хмельницька область за 1982 рік:
   Week    VHI
0   1.0  49.95
1   2.0  47.04
2   3.0  44.99
3   4.0  41.29
4   5.0  37.72
5   6.0  34.91
6   7.0  33.14
7   8.0  32.72
8   9.0  32.77
9  10.0  32.23


## Завдання №4.2
Ряд VHI за вказаний діапазон років для вказаних областей

In [6]:
def get_vhi_range(dataframe,province_ids,year_start,year_end):
    condition1=dataframe['new_province_id'].isin(province_ids)
    condition2=dataframe['Year']>=year_start
    condition3=dataframe['Year']<=year_end
    filtered_data=dataframe[condition1&condition2&condition3]
    return filtered_data[['Year', 'Week', 'VHI', 'new_province_name']]
print("Хмельницька та Київська області за 1982-1985 роки:")
result=get_vhi_range(df, [21, 9], 1982, 1985)
print(result.head(5))

Хмельницька та Київська області за 1982-1985 роки:
   Year  Week    VHI new_province_name
0  1982   1.0  49.95       Хмельницька
1  1982   2.0  47.04       Хмельницька
2  1982   3.0  44.99       Хмельницька
3  1982   4.0  41.29       Хмельницька
4  1982   5.0  37.72       Хмельницька


## Завдання №4.3
Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани

In [7]:
def get_vhi_statistics(dataframe, province_ids, years):
    condition1 = dataframe['new_province_id'].isin(province_ids)
    condition2 = dataframe['Year'].isin(years)
    filtered_data = dataframe[condition1 & condition2]
    print("Екстремум(макс.):")
    vhi_min = filtered_data['VHI'].min()
    print(vhi_min)
    print("Екстремум(мін.):")
    vhi_max = filtered_data['VHI'].max()
    print(vhi_max)
    print("Середнє арифметичне:")
    vhi_mean = filtered_data['VHI'].mean()
    print(vhi_mean)
    print("Медіана:")
    vhi_median = filtered_data['VHI'].median()
    print(vhi_median)
print("Статистика для Хмельницької (21) та Київської (9) областей за 1982 та 1985 роки:")
get_vhi_statistics(df, [21, 9], [1982, 1985])

Статистика для Хмельницької (21) та Київської (9) областей за 1982 та 1985 роки:
Екстремум(макс.):
-1.0
Екстремум(мін.):
63.72
Середнє арифметичне:
38.56076923076923
Медіана:
39.555
